# AI-Inferred 3D Video Reconstruction — Google Colab

This notebook runs the same shared `src/`, `blender/`, and `config/` pipeline used by the local application. It does not start the browser UI or duplicate reconstruction logic.

Before running: push the latest project changes to GitHub, select **Runtime → Change runtime type → GPU**, then run the cells in order. Completed Blender gaps and final results are copied to Google Drive. Colab hardware and session lifetime are not guaranteed.

In [ ]:
from pathlib import Path

REPOSITORY_URL = "https://github.com/redmas45/3D_reconstruction.git"
REPOSITORY_BRANCH = "main"
PROJECT_DIRECTORY = Path("/content/3D_reconstruction")
DRIVE_ROOT = Path("/content/drive/MyDrive/3D_Reconstruction")
BLENDER_VERSION = "4.5.10"
BLENDER_ARCHIVE_URL = (
    f"https://download.blender.org/release/Blender4.5/"
    f"blender-{BLENDER_VERSION}-linux-x64.tar.xz"
)
RANDOM_SEED = 42
MAX_PARALLEL_GAP_RENDERERS = 2
RENDERER_MODE = "blender"

print("Notebook configuration loaded.")

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA GPU was assigned. Change the Colab runtime type to GPU and reconnect.")

subprocess.run(["nvidia-smi"], check=True)


In [ ]:
import os

if (PROJECT_DIRECTORY / ".git").is_dir():
    subprocess.run(["git", "fetch", "origin", REPOSITORY_BRANCH], cwd=PROJECT_DIRECTORY, check=True)
    subprocess.run(["git", "checkout", REPOSITORY_BRANCH], cwd=PROJECT_DIRECTORY, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH], cwd=PROJECT_DIRECTORY, check=True)
elif PROJECT_DIRECTORY.exists() and any(PROJECT_DIRECTORY.iterdir()):
    raise RuntimeError(f"Refusing to overwrite non-repository directory: {PROJECT_DIRECTORY}")
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(PROJECT_DIRECTORY)],
        check=True,
    )

os.chdir(PROJECT_DIRECTORY)
print(f"Project ready at {PROJECT_DIRECTORY}")


In [ ]:
import shlex
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True,
)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "ffmpeg", "xvfb", "xauth", "libgl1", "libxi6", "libxrender1"],
    check=True,
)

blender_install_root = Path("/content/blender")
blender_binary = blender_install_root / f"blender-{BLENDER_VERSION}-linux-x64" / "blender"
if not blender_binary.is_file():
    blender_install_root.mkdir(parents=True, exist_ok=True)
    archive_path = blender_install_root / "blender.tar.xz"
    subprocess.run(["curl", "--fail", "--location", BLENDER_ARCHIVE_URL, "--output", str(archive_path)], check=True)
    subprocess.run(["tar", "-xf", str(archive_path), "-C", str(blender_install_root)], check=True)

if not blender_binary.is_file():
    raise FileNotFoundError(f"Blender executable was not installed at {blender_binary}")

blender_wrapper = blender_install_root / "blender-colab"
blender_wrapper.write_text(
    "#!/usr/bin/env bash\nexec xvfb-run -a " + shlex.quote(str(blender_binary)) + " \"$@\"\n",
    encoding="utf-8",
)
blender_wrapper.chmod(0o755)
os.environ["BLENDER_EXECUTABLE"] = str(blender_wrapper)
subprocess.run([str(blender_wrapper), "--background", "--version"], check=True)
subprocess.run(["ffmpeg", "-version"], check=True, stdout=subprocess.DEVNULL)
print("Python dependencies, FFmpeg, and Blender are ready.")


In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

uploaded_files = files.upload()
if len(uploaded_files) != 1:
    raise ValueError("Upload exactly one video file.")

uploaded_name, uploaded_bytes = next(iter(uploaded_files.items()))
safe_name = Path(uploaded_name).name
supported_extensions = {".mp4", ".mov", ".avi", ".mkv", ".m4v", ".webm", ".mpeg", ".mpg", ".wmv"}
if Path(safe_name).suffix.lower() not in supported_extensions:
    raise ValueError(f"Unsupported video extension: {Path(safe_name).suffix}")

input_directory = Path("/content/reconstruction_input")
input_directory.mkdir(parents=True, exist_ok=True)
input_video = input_directory / safe_name
input_video.write_bytes(uploaded_bytes)
del uploaded_bytes, uploaded_files
print(f"Input ready: {input_video.name} ({input_video.stat().st_size / (1024 ** 2):.1f} MB)")


In [ ]:
import copy
import hashlib
import random
import threading
from collections import deque

import ipywidgets as widgets
from IPython.display import display

source_directory = PROJECT_DIRECTORY / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

from application.reconstruction_pipeline import PipelineOptions, load_config, process_video

with input_video.open("rb") as input_stream:
    video_digest = hashlib.file_digest(input_stream, "sha256").hexdigest()[:12]
run_identifier = f"{input_video.stem}_{video_digest}"
local_output = Path("/content/reconstruction_runs") / run_identifier
drive_run = DRIVE_ROOT / run_identifier
drive_checkpoint_gaps = drive_run / "checkpoints" / "gaps"
local_gaps = local_output / "_work" / input_video.stem / "gaps"

if drive_checkpoint_gaps.is_dir():
    shutil.copytree(drive_checkpoint_gaps, local_gaps, dirs_exist_ok=True)
    print(f"Restored completed-gap checkpoints from {drive_checkpoint_gaps}")

configuration = copy.deepcopy(load_config(PROJECT_DIRECTORY / "config" / "reconstruction_config.json"))
configuration["renderer"]["max_parallel_gap_renders"] = MAX_PARALLEL_GAP_RENDERERS
options = PipelineOptions(configuration, local_output, reuse_work=True, renderer_mode=RENDERER_MODE)

progress_bar = widgets.FloatProgress(value=0, min=0, max=100, description="Progress", bar_style="info")
stage_label = widgets.HTML(value="<b>Waiting to start</b>")
activity_output = widgets.Output(layout={"border": "1px solid #475569", "max_height": "260px", "overflow_y": "auto"})
display(widgets.VBox([stage_label, progress_bar, activity_output]))

activity_lock = threading.Lock()
checkpoint_lock = threading.Lock()
activity_lines = deque(maxlen=16)
checkpointed_gaps: set[str] = set()
last_displayed_percentage = -1

def checkpoint_completed_gaps() -> None:
    if not local_gaps.is_dir():
        return
    with checkpoint_lock:
        for gap_directory in sorted(local_gaps.glob("gap_*")):
            blender_directory = gap_directory / "blender"
            required = [blender_directory / "gap_blender.mp4", blender_directory / "render_report.json", blender_directory / "scene.blend"]
            if gap_directory.name in checkpointed_gaps or not all(path.is_file() for path in required):
                continue
            shutil.copytree(gap_directory, drive_checkpoint_gaps / gap_directory.name, dirs_exist_ok=True)
            checkpointed_gaps.add(gap_directory.name)

def watch_for_checkpoints(stop_event: threading.Event) -> None:
    while not stop_event.wait(10):
        checkpoint_completed_gaps()

def report_progress(stage: str, progress: float, detail: str) -> None:
    global last_displayed_percentage
    percentage = round(progress * 100)
    with activity_lock:
        progress_bar.value = percentage
        stage_label.value = f"<b>{stage.replace('_', ' ').title()}</b> — {detail}"
        if percentage != last_displayed_percentage:
            activity_lines.append(f"[{percentage:3d}%] {detail}")
            last_displayed_percentage = percentage
            with activity_output:
                activity_output.clear_output(wait=True)
                print("\n".join(activity_lines))

checkpoint_stop_event = threading.Event()
checkpoint_thread = threading.Thread(
    target=watch_for_checkpoints, args=(checkpoint_stop_event,), name="drive-checkpoint", daemon=True
)
checkpoint_thread.start()
print(f"Running {RENDERER_MODE} reconstruction with up to {MAX_PARALLEL_GAP_RENDERERS} Blender workers.")
try:
    final_output = process_video(input_video, options, random.Random(RANDOM_SEED), report_progress)
finally:
    checkpoint_stop_event.set()
    checkpoint_thread.join(timeout=15)
    checkpoint_completed_gaps()
print(f"Reconstruction finished: {final_output}")


In [ ]:
from IPython.display import Video

result_directory = drive_run / "result"
result_directory.mkdir(parents=True, exist_ok=True)
drive_video = result_directory / final_output.name
shutil.copy2(final_output, drive_video)

report_directory = result_directory / "reports"
report_directory.mkdir(parents=True, exist_ok=True)
work_directory = local_output / "_work" / input_video.stem
for report_path in work_directory.rglob("*.json"):
    relative_path = report_path.relative_to(work_directory)
    destination = report_directory / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(report_path, destination)

print(f"Saved final video to Google Drive: {drive_video}")
display(Video(str(final_output), embed=True, width=960))
files.download(str(final_output))


## Troubleshooting

- If the GPU check fails, reconnect using a GPU runtime.
- If cloning does not include recent changes, push the local branch to GitHub and rerun the clone cell.
- If Colab disconnects, reconnect and run the cells again with the same video and seed. Completed Blender gap checkpoints will be restored from Drive.
- If Blender exhausts GPU or system memory, change `MAX_PARALLEL_GAP_RENDERERS` from `2` to `1`.
- The output is an AI-inferred evidence visualization, not recovered ground truth.